

# YouTube Summarizer — Paste a Link, Get Key Takeaways

**Project | AI Mini Projects Series**

**Problem:** Long YouTube videos take too much time to watch just to find the main points.

**What this notebook does:** You paste a YouTube link, it downloads the video transcript, and an LLM gives you a short summary and the key takeaways.

**How it works:** YouTube link → video id → transcript text → split long text into parts → summarize each part → combine into final summary + takeaways.

**Tools:** Python, youtube-transcript-api, Groq (openai/gpt-oss-120b), Google Colab

## Step 1: Install libraries

In [1]:
!pip install -q youtube-transcript-api openai scikit-learn

## Step 2: Import libraries and enter Groq key

In [2]:
import os
import re
import getpass
from openai import OpenAI

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)
model_name = "openai/gpt-oss-120b"
print("Groq client ready")

Enter your Groq API key: ··········
Groq client ready


## Step 3: Helper function to call the LLM

In [3]:
def ask_llm(system_msg, user_msg):
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": user_msg}
            ],
            temperature=0.3
        )
        return response.choices[0].message.content
    except Exception as e:
        return "Error calling Groq: " + str(e)

## Step 4: Get the video id from the YouTube link
Works with normal links, youtu.be short links, shorts and embed links.

In [4]:
def get_video_id(link):
    link = link.strip()
    patterns = [
        r"v=([A-Za-z0-9_-]{11})",
        r"youtu\.be/([A-Za-z0-9_-]{11})",
        r"shorts/([A-Za-z0-9_-]{11})",
        r"embed/([A-Za-z0-9_-]{11})"
    ]
    for p in patterns:
        match = re.search(p, link)
        if match:
            return match.group(1)
    return None

# quick test
print(get_video_id("https://www.youtube.com/watch?v=dQw4w9WgXcQ"))
print(get_video_id("https://youtu.be/dQw4w9WgXcQ"))

dQw4w9WgXcQ
dQw4w9WgXcQ


## Step 5: Get the transcript
This tries the transcript download first. YouTube sometimes blocks Colab, so if it fails you can paste the transcript yourself, or the sample transcript is used.

In [5]:
def get_transcript(video_id):
    text = ""
    try:
        from youtube_transcript_api import YouTubeTranscriptApi
        try:
            api = YouTubeTranscriptApi()
            data = api.fetch(video_id, languages=["en", "hi"])
            for item in data:
                text = text + item.text + " "
        except AttributeError:
            text = ""
            data = YouTubeTranscriptApi.get_transcript(video_id, languages=["en", "hi"])
            for item in data:
                text = text + item["text"] + " "
    except Exception as e:
        print("Could not get transcript:", str(e)[:200])
        text = ""
    return text.strip()

sample_transcript = (
    "Hi everyone, today I want to talk about how to build a daily study habit that actually sticks. "
    "Most people fail because they try to study for four hours on day one and quit by day three. "
    "The first tip is to start very small. Just study for twenty minutes every day at the same time. "
    "When the time is fixed, your brain starts expecting it and you need less willpower. "
    "The second tip is to remove distractions. Keep your phone in another room and use only one tab on your laptop. "
    "The third tip is to track your progress. Use a simple calendar and mark a cross on every day you study. "
    "After a week you will not want to break the chain. "
    "The fourth tip is to use active recall. Do not just read the notes again and again. Close the book and try to write what you remember. "
    "This feels harder but it makes the memory much stronger. "
    "The fifth tip is to take proper breaks. Study for twenty five minutes, then rest for five minutes. After four rounds take a longer break. "
    "Also sleep at least seven hours because the brain saves what you learned during sleep. "
    "Finally, do not compare yourself with others. Compare yourself with who you were last week. "
    "If you follow these five tips for thirty days, studying will become a normal part of your day. Thanks for watching."
)

link = input("Paste a YouTube link (or press Enter for the sample): ")
transcript = ""
using_sample = False

if link.strip() != "":
    video_id = get_video_id(link)
    if video_id is None:
        print("Could not find a video id in that link.")
    else:
        print("Video id:", video_id)
        transcript = get_transcript(video_id)

if transcript == "" and link.strip() != "":
    print("Transcript download did not work (YouTube may block Colab).")
    pasted = input("Paste the transcript text here, or press Enter to use the sample: ")
    transcript = pasted.strip()

if transcript == "":
    transcript = sample_transcript
    using_sample = True
    print("Using the sample transcript.")

print("Transcript length (characters):", len(transcript))
print("First 300 characters:")
print(transcript[:300])

Paste a YouTube link (or press Enter for the sample): https://youtu.be/4Vz6L8B73i4?si=LKud1E31Tlpuui5D
Video id: 4Vz6L8B73i4
Transcript length (characters): 164332
First 300 characters:
If you let people have their phones out, the IQ of your employees is lower. Cuz there was a study done when the phone was visible to you. Scores were significantly decreased. >> Like my attention decreased. >> Yes, Raj. It was also general fluid intelligence. That means you're dumber. This is the sc


## Step 6: Split long transcripts into parts
Long videos have a lot of text, so we split it and summarize each part first.

In [6]:
def split_text(text, size=6000):
    parts = []
    start = 0
    while start < len(text):
        end = start + size
        if end < len(text):
            # try to cut at a space so we do not break a word
            space_pos = text.rfind(" ", start, end)
            if space_pos > start:
                end = space_pos
        parts.append(text[start:end])
        start = end
    return parts

parts = split_text(transcript)
print("Number of parts:", len(parts))

Number of parts: 28


## Step 7: Summarize each part, then combine

In [7]:
import time

def summarize_part(text):
    for attempt in range(3):
        s = ask_llm(
            "You summarize parts of a video transcript. Write 4 to 5 short lines covering the main points only.",
            text
        )
        if not s.startswith("Error calling Groq"):
            return s
        print("  Groq error, waiting and trying again (try", attempt + 1, "of 3)")
        time.sleep(10 * (attempt + 1))
    return None

part_summaries = []
failed_parts = []

if len(parts) == 1:
    part_summaries.append(parts[0])
else:
    for i in range(len(parts)):
        print("Summarizing part", i + 1, "of", len(parts))
        s = summarize_part(parts[i])
        if s is None:
            failed_parts.append(i + 1)
            s = "(part " + str(i + 1) + " could not be summarized)"
        part_summaries.append(s)

if len(failed_parts) > 0:
    print("WARNING: these parts failed:", failed_parts)
    print("The summary below may miss them. Run this cell again to retry.")

all_text = "\n\n".join(part_summaries)

final_prompt = (
    "Here is the transcript (or summaries of its parts) of a YouTube video.\n\n" + all_text + "\n\n"
    "Give the answer in this format:\n"
    "SUMMARY: 3 to 4 simple sentences\n"
    "KEY TAKEAWAYS: 5 short bullet points\n"
    "WHO SHOULD WATCH: one line"
)

final_answer = ask_llm(
    "You are a helpful assistant that writes short and clear video summaries in simple English.",
    final_prompt
)

print(final_answer)

Summarizing part 1 of 28
Summarizing part 2 of 28
Summarizing part 3 of 28
Summarizing part 4 of 28
Summarizing part 5 of 28
Summarizing part 6 of 28
Summarizing part 7 of 28
Summarizing part 8 of 28
Summarizing part 9 of 28
Summarizing part 10 of 28
Summarizing part 11 of 28
Summarizing part 12 of 28
Summarizing part 13 of 28
Summarizing part 14 of 28
Summarizing part 15 of 28
Summarizing part 16 of 28
Summarizing part 17 of 28
Summarizing part 18 of 28
Summarizing part 19 of 28
Summarizing part 20 of 28
Summarizing part 21 of 28
Summarizing part 22 of 28
Summarizing part 23 of 28
Summarizing part 24 of 28
Summarizing part 25 of 28
Summarizing part 26 of 28
Summarizing part 27 of 28
Summarizing part 28 of 28
**SUMMARY:**  
Seeing your phone or trying to multitask makes you less smart, slower, and less trustworthy.  Every time you switch attention you lose time and energy, so top performers protect their focus, embrace boredom, and give their brain space to create ideas.  You can train

## Step 8: Build a search index over the full transcript
For follow-up questions we search the full transcript, not only the summaries, so details and the end of a long video are not missed.

In [8]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

stop_list = ["the", "a", "an", "is", "are", "do", "does", "you", "i", "we", "of", "to", "for", "and", "in", "on", "at", "it", "my", "your", "what", "how", "can", "with", "this", "that", "me", "us", "there", "any", "have", "has", "be", "or", "she", "he", "they", "say", "says", "video"]

def stem_tokens(text):
    # cut every word to 5 letters so focus / focused / focusing match
    words = re.findall(r"[a-z0-9]+", text.lower())
    result = []
    for w in words:
        if w not in stop_list:
            result.append(w[:5])
    return result

search_chunks = split_text(transcript, 2500)
vectorizer = TfidfVectorizer(tokenizer=stem_tokens, lowercase=False, token_pattern=None)
search_matrix = vectorizer.fit_transform(search_chunks)

def find_parts(question, top_k=4):
    q_vec = vectorizer.transform([question])
    scores = linear_kernel(q_vec, search_matrix).flatten()
    top_ids = scores.argsort()[::-1][:top_k]
    found = []
    for i in sorted(top_ids):
        if scores[i] > 0:
            found.append(search_chunks[i])
    return found

print("Search chunks:", len(search_chunks))

Search chunks: 66


## Step 9: Ask a follow-up question about the video
Type a question about the video. Type `exit` to stop.

In [9]:
while True:
    q = input("Ask something about the video (type exit to stop): ")
    if q.strip().lower() == "exit":
        print("Bye!")
        break
    if q.strip() == "":
        continue

    found = find_parts(q)
    context = "Short summary of the video:\n" + final_answer + "\n\nParts of the transcript that match the question:\n"
    for f in found:
        context = context + "..." + f + "...\n\n"

    a = ask_llm(
        "Answer the question only using the video text given. If it is not in the text, say 'This is not mentioned in the video'. Keep it short.",
        context + "\nQuestion: " + q
    )
    print("Answer:", a)
    print("(matched", len(found), "transcript parts)")
    print("-" * 60)

Ask something about the video (type exit to stop): What does she say are the main reasons people struggle to focus today?
Answer: She says people today lose focus mainly because of constant digital distractions—phones and notifications that pull attention away—and because they’re caught in habit loops and addictive scrolling, often reinforced by limiting beliefs or a “victim” mindset that sees obstacles instead of focusing on the task.
(matched 4 transcript parts)
------------------------------------------------------------
Ask something about the video (type exit to stop): How does she suggest improving focus and concentration?
Answer: She says to protect focus by removing digital distractions (keep the phone hidden or off and turn off notifications), use tiny “if‑then” habits and clear identity statements, schedule regular micro‑breaks (and weekly/monthly breaks), work during your natural energy‑peak times, and embrace boredom to give your brain space for ideas. These steps together 

QUE


What does she say are the main reasons people struggle to focus today?

How does she suggest improving focus and concentration?

What does she say about how memory works in the brain?

What techniques or habits does she recommend for remembering things better?

How do sleep, diet, or exercise affect focus and memory, according to the video?

What does she say about phones, social media, or distractions?

Does she mention any daily routine or practical tips to follow?

What does she say about stress, anxiety, or mental health in relation to focus?

What are the key takeaways or final advice from the conversation?

## Key Points

- The notebook works on the transcript text, so videos without captions cannot be summarized (you can paste the text yourself)
- Long videos are summarized part by part first, then combined, so the text does not get too big for the model
- If Groq gives an error on a part, the notebook waits and retries, and warns you if a part still fails, so an error message never ends up inside the summary
- Follow-up questions search the full transcript (not only the summaries), so details and the end of a long video are covered. If nothing matches, the bot says "This is not mentioned in the video"
- Sometimes YouTube blocks transcript downloads from Colab. That is why there is a paste option and a sample transcript, so the notebook never gets stuck
- The summary is only as good as the transcript. Auto-generated captions can have mistakes, and the video may contain claims that are not proven, so treat it as a summary of what was said, not as facts

**Ideas to improve:** save summaries to a file, add timestamps for each takeaway, support more languages, make a Streamlit app.

---
Built by Akshat Kesharwani | GitHub: https://github.com/akshatkesharwani-info | Portfolio: https://akshatkesharwani-info.github.io/